In [ ]:

### **Clase de Físicas: Integrando PyMunk**

**Objetivo:** Extender el motor con un sistema de físicas 2D opcional usando PyMunk. Crear una escena de demostración interactiva (sandbox) para probar colisiones, gravedad y manipulación de objetos con el ratón.

-----

### **1. El Concepto: Composición y "Mixins"**

No todos los objetos en un juego necesitan físicas. Un botón del menú o un icono en la interfaz no deben caer por la gravedad. Por eso, no modificaremos nuestra clase base `GameObject`.
En su lugar, crearemos un "kit de mejora" llamado `PhysicsMixin`.

  * **`GameObject`:** Sigue siendo la base de todo lo visible. Tiene una imagen y una posición.
  * **`PhysicsMixin`:** Contiene toda la lógica para crear un cuerpo físico (`Body`) y una forma de colisión (`Shape`) en PyMunk.
  No es un objeto por sí mismo, sino un conjunto de capacidades.

Para crear un objeto con físicas, haremos que herede de **ambas** clases.
Así, un `PhysicsCircle` será un `GameObject` (para poder ser dibujado) **y** un `PhysicsMixin` (para poder colisionar y caer).

-----

### **2. Expandiendo el Motor**

Añadimos un nuevo archivo a nuestra carpeta `engine/`.
Agregar al requirements.txt: pymunk

Activar el entorno virtual e instalar las librerías faltantes:

Ej:

PS C:\TVJ\clase_12_física> .\venv\Scripts\activate
(venv) PS C:\TVJ\clase_12_física> pip install -r .\requirements.txt


#### **`engine/physics.py` (¡NUEVO\!)**

Este módulo contiene nuestro "kit de físicas".

```python
# engine/physics.py
import pygame
import pymunk

class PhysicsMixin:
    """
    Un 'mixin' que añade propiedades y métodos de PyMunk a un GameObject.
    No está pensado para ser usado solo, sino en herencia múltiple.
    """
    def init_physics_circle(self, space, mass, radius, elasticity, friction):
        """Inicializa un cuerpo físico circular."""
        moment = pymunk.moment_for_circle(mass, 0, radius)
        self.body = pymunk.Body(mass, moment)
        self.body.position = self.rect.center
        self.shape = pymunk.Circle(self.body, radius)
        self.shape.elasticity = elasticity
        self.shape.friction = friction
        self.shape.parent = self # Guardamos una referencia al objeto del juego
        space.add(self.body, self.shape)

    def init_physics_box(self, space, mass, elasticity, friction):
        """Inicializa un cuerpo físico rectangular."""
        size = self.rect.size
        moment = pymunk.moment_for_box(mass, size)
        self.body = pymunk.Body(mass, moment)
        self.body.position = self.rect.center
        self.shape = pymunk.Poly.create_box(self.body, size)
        self.shape.elasticity = elasticity
        self.shape.friction = friction
        self.shape.parent = self
        space.add(self.body, self.shape)

    def update_from_physics(self):
        """Sincroniza la posición y rotación del sprite con el cuerpo físico."""
        self.rect.center = self.body.position
        # PyMunk usa radianes, Pygame usa grados. Hacemos la conversión para la rotación.
        self.image_rotated = pygame.transform.rotate(self.original_image, -math.degrees(self.body.angle))
        self.image = self.image_rotated
        self.rect = self.image.get_rect(center=self.rect.center)
```

-----

### **3. El Juego de Ejemplo: "Sandbox de Físicas"**

Crearemos un nuevo proyecto para probar nuestro motor de físicas.

#### **`physics_sandbox/entities.py`**

Aquí creamos nuestras figuras geométricas, que heredan de `GameObject` y `PhysicsMixin`.

```python
import pygame
import math # Necesario para la rotación
from engine.game_object import GameObject
from engine.physics import PhysicsMixin

class PhysicsCircle(GameObject, PhysicsMixin):
    def __init__(self, x, y, radius, space):
        # Creamos una imagen circular dinámicamente
        image = pygame.Surface((radius * 2, radius * 2), pygame.SRCALPHA)
        pygame.draw.circle(image, (200, 30, 30), (radius, radius), radius)

        super().__init__(x, y, image)
        self.original_image = self.image # Para la rotación

        # Usamos el mixin para inicializar el cuerpo físico
        self.init_physics_circle(space, mass=10, radius=radius, elasticity=0.6, friction=0.8)

    def update(self):
        self.update_from_physics() # Sincronizamos con la física

class PhysicsBox(GameObject, PhysicsMixin):
    def __init__(self, x, y, width, height, space):
        image = pygame.Surface((width, height), pygame.SRCALPHA)
        image.fill((30, 30, 200))

        super().__init__(x, y, image)
        self.original_image = self.image

        self.init_physics_box(space, mass=15, elasticity=0.4, friction=0.9)

    def update(self):
        self.update_from_physics()
```

#### **`physics_sandbox/game.py`**

Esta es la escena principal, que contiene toda la lógica de la simulación y la interacción con el ratón.

```python
import pygame
import pymunk
from entities import PhysicsCircle, PhysicsBox
from game_loop import GameLoop


class PhysicsGame(GameLoop):
    def __init__(self):
        super().__init__(1280, 720, "Sandbox de Físicas con PyMunk", 60)

        # 1. Configuración del Espacio de Físicas
        self.space = pymunk.Space()
        self.space.gravity = (0, 900)  # Gravedad hacia abajo

        self.sprites = pygame.sprite.Group()
        self.create_boundaries()

        # --- AÑADIDO: Creamos objetos al inicio para que la pantalla no esté vacía ---
        self.crear_objetos_iniciales()
        # -------------------------------------------------------------------------

        # 2. Lógica para arrastrar con el ratón
        self.mouse_body = pymunk.Body(body_type=pymunk.Body.KINEMATIC)
        self.mouse_joint = None
        self.mouse_dragging = False

    def crear_objetos_iniciales(self):
        """Crea algunas figuras al inicio para que el sandbox no esté vacío."""
        caja1 = PhysicsBox(600, 600, 50, 50, self.space)
        caja2 = PhysicsBox(600, 550, 50, 50, self.space)
        circulo1 = PhysicsCircle(650, 600, 30, self.space)
        self.sprites.add(caja1, caja2, circulo1)

    def create_boundaries(self):
        """Crea el suelo y las paredes estáticas."""
        # El suelo está 5 píxeles por debajo para que se vean los objetos
        floor = pymunk.Segment(self.space.static_body, (0, self.screen.get_height() - 5),
                               (self.screen.get_width(), self.screen.get_height() - 5), 5)
        floor.elasticity = 0.8
        floor.friction = 1.0

        wall_left = pymunk.Segment(self.space.static_body, (5, 0), (5, self.screen.get_height()), 5)
        wall_right = pymunk.Segment(self.space.static_body, (self.screen.get_width() - 5, 0),
                                    (self.screen.get_width() - 5, self.screen.get_height()), 5)

        self.space.add(floor, wall_left, wall_right)

    def handle_specific_events(self, event):
        if event.type == pygame.KEYDOWN:

            # --- ¡ESTA ES LA CORRECCIÓN! ---
            # Obtenemos la posición del ratón por separado usando pygame.mouse.get_pos()
            # El evento de teclado (KEYDOWN) no tiene el atributo .pos
            pos = pygame.mouse.get_pos()

            if event.key == pygame.K_c:  # 'c' para crear un círculo
                circle = PhysicsCircle(pos[0], pos[1], 30, self.space)
                self.sprites.add(circle)
            if event.key == pygame.K_b:  # 'b' para crear una caja (box)
                box = PhysicsBox(pos[0], pos[1], 50, 50, self.space)
                self.sprites.add(box)
            # ---------------------------------

        # --- Lógica de Agarrar y Soltar ---
        elif event.type == pygame.MOUSEBUTTONDOWN and event.button == 1:
            point_query = self.space.point_query_nearest(event.pos, 0, pymunk.ShapeFilter())
            if point_query and point_query.shape.body.body_type == pymunk.Body.DYNAMIC:
                shape = point_query.shape
                self.mouse_joint = pymunk.PivotJoint(self.mouse_body, shape.body, shape.body.world_to_local(event.pos))
                self.mouse_joint.max_force = 70000  # Aumentamos la fuerza para que no se "rompa"
                self.space.add(self.mouse_joint)
                self.mouse_dragging = True

        elif event.type == pygame.MOUSEMOTION:
            self.mouse_body.position = event.pos

        elif event.type == pygame.MOUSEBUTTONUP and event.button == 1:
            if self.mouse_dragging:
                self.space.remove(self.mouse_joint)
                self.mouse_joint = None
                self.mouse_dragging = False

    def update_game_logic(self):
        # 1. Avanzamos la simulación de físicas
        dt = 1.0 / self.fps
        self.space.step(dt)

        # 2. Actualizamos los sprites para que sigan a sus cuerpos físicos
        self.sprites.update()

    def draw_game_elements(self):
        self.screen.fill((210, 220, 230))  # Fondo gris claro
        self.sprites.draw(self.screen)
```

#### **`physics_sandbox/main.py`**

```python
# main.py
from .game import PhysicsGame

if __name__ == '__main__':
    juego = PhysicsGame()
    juego.run()
```

-----

### **4. Paso a Paso del Funcionamiento**

1.  **Haces clic en un objeto:** El código usa `space.point_query_nearest` para encontrar la figura de PyMunk más cercana a tu cursor.
2.  **Se crea una "cuerda" invisible:** Se crea una unión (`PivotJoint`) entre un cuerpo invisible que sigue a tu ratón (`mouse_body`) y el cuerpo del objeto que has agarrado.
3.  **Arrastras:** Mueves el ancla (`mouse_body`) con tu ratón, y la "cuerda" tira del objeto, aplicando fuerzas realistas.
4.  **Sueltas:** Se corta la cuerda (`space.remove(self.mouse_joint)`), y el objeto sale disparado con la inercia que le hayas dado.



Aseprite para dibujar sprites
Sketchbook para dibujar